In [22]:
from pathlib import Path
from typing import Dict, List

import torch


LOG_ROOT = Path("/workspace/code/logs/train")
MODEL_RUNS = {
    "ssm": LOG_ROOT / "ssm" / "2026-04-17" / "12-12-47" / "fold_0" / "test_analysis",
    "st_gcn": LOG_ROOT / "st_gcn" / "2026-04-17" / "12-12-30" / "fold_0" / "test_analysis",
    "tcn": LOG_ROOT / "tcn" / "2026-04-17" / "12-13-00" / "fold_0" / "test_analysis",
}
TASKS = ["twist", "posture", "relax", "total"]
NUM_FOLDS = 10


def compute_macro_f1(pred_label: torch.Tensor, true_label: torch.Tensor, num_classes: int) -> float:
    f1_scores: List[float] = []
    for class_idx in range(num_classes):
        tp = ((pred_label == class_idx) & (true_label == class_idx)).sum().item()
        fp = ((pred_label == class_idx) & (true_label != class_idx)).sum().item()
        fn = ((pred_label != class_idx) & (true_label == class_idx)).sum().item()
        denominator = 2 * tp + fp + fn
        f1_scores.append((2 * tp / denominator) if denominator > 0 else 0.0)
    return float(sum(f1_scores) / len(f1_scores))


def load_fold_task_result(model_dir: Path, task: str, fold: int) -> Dict[str, float]:
    pred_path = model_dir / task / "best_preds" / f"fold_{fold}_pred.pt"
    label_path = model_dir / task / "best_preds" / f"fold_{fold}_label.pt"
    if not pred_path.exists() or not label_path.exists():
        raise FileNotFoundError(f"Missing prediction files for {task} fold_{fold}: {pred_path} / {label_path}")

    pred = torch.load(pred_path, map_location="cpu")
    label = torch.load(label_path, map_location="cpu").long().view(-1)

    if pred.ndim == 2:
        pred_label = pred.argmax(dim=1)
        num_classes = pred.shape[1]
    else:
        pred_label = pred.long().view(-1)
        num_classes = int(max(pred_label.max().item(), label.max().item()) + 1)

    accuracy = (pred_label == label).float().mean().item()
    macro_f1 = compute_macro_f1(pred_label, label, num_classes=num_classes)
    return {
        "acc": float(accuracy),
        "f1": float(macro_f1),
        "num_samples": int(label.numel()),
    }


def build_result_rows(model_runs: Dict[str, Path], tasks: List[str], num_folds: int) -> List[Dict[str, object]]:
    rows: List[Dict[str, object]] = []
    for model_name, model_dir in model_runs.items():
        for fold in range(num_folds):
            task_metrics: Dict[str, Dict[str, float]] = {}
            try:
                for task in tasks:
                    task_metrics[task] = load_fold_task_result(model_dir, task, fold)
            except FileNotFoundError:
                continue

            avg_acc = sum(task_metrics[task]["acc"] for task in tasks) / len(tasks)
            avg_f1 = sum(task_metrics[task]["f1"] for task in tasks) / len(tasks)
            rows.append(
                {
                    "model": model_name,
                    "fold": fold,
                    "avg_acc": float(avg_acc),
                    "avg_f1": float(avg_f1),
                    "tasks": task_metrics,
                }
            )
    return rows


def build_fold_ranking(rows: List[Dict[str, object]], num_folds: int) -> Dict[int, List[Dict[str, object]]]:
    ranking: Dict[int, List[Dict[str, object]]] = {}
    for fold in range(num_folds):
        items = [row for row in rows if row["fold"] == fold]
        ranking[fold] = sorted(items, key=lambda item: item["avg_f1"], reverse=True)
    return ranking


def build_model_summary(rows: List[Dict[str, object]]) -> List[Dict[str, float]]:
    summary: List[Dict[str, float]] = []
    for model_name in sorted({row["model"] for row in rows}):
        model_rows = [row for row in rows if row["model"] == model_name]
        if not model_rows:
            continue
        summary.append(
            {
                "model": model_name,
                "mean_f1": float(sum(row["avg_f1"] for row in model_rows) / len(model_rows)),
                "mean_acc": float(sum(row["avg_acc"] for row in model_rows) / len(model_rows)),
                "num_folds": len(model_rows),
            }
        )
    return sorted(summary, key=lambda item: item["mean_f1"], reverse=True)


def task_level_summary(rows: List[Dict[str, object]], tasks: List[str]) -> Dict[str, List[Dict[str, float]]]:
    summary: Dict[str, List[Dict[str, float]]] = {}
    for task in tasks:
        task_rows: List[Dict[str, float]] = []
        for model_name in sorted({row["model"] for row in rows}):
            model_rows = [row for row in rows if row["model"] == model_name]
            if not model_rows:
                continue
            task_rows.append(
                {
                    "model": model_name,
                    "mean_f1": float(sum(row["tasks"][task]["f1"] for row in model_rows) / len(model_rows)),
                    "mean_acc": float(sum(row["tasks"][task]["acc"] for row in model_rows) / len(model_rows)),
                }
            )
        summary[task] = sorted(task_rows, key=lambda item: item["mean_f1"], reverse=True)
    return summary


def format_fold_ranking(fold_ranking: Dict[int, List[Dict[str, object]]]) -> List[str]:
    lines: List[str] = []
    for fold, items in fold_ranking.items():
        if not items:
            lines.append(f"fold_{fold}: no results")
            continue
        one_line = " | ".join(
            [
                f"{item['model']}: f1={item['avg_f1']:.4f}, acc={item['avg_acc']:.4f}"
                for item in items
            ]
        )
        lines.append(f"fold_{fold}: {one_line}")
    return lines


def format_top_k(rows: List[Dict[str, object]], top_k: int = 3) -> List[str]:
    sorted_rows = sorted(rows, key=lambda item: item["avg_f1"], reverse=True)[:top_k]
    return [
        f"Top {rank}: {item['model']} fold_{item['fold']} | avg_f1={item['avg_f1']:.4f} | avg_acc={item['avg_acc']:.4f}"
        for rank, item in enumerate(sorted_rows, start=1)
    ]


def format_model_summary(summary_rows: List[Dict[str, float]]) -> List[str]:
    return [
        f"{item['model']}: mean_f1={item['mean_f1']:.4f}, mean_acc={item['mean_acc']:.4f}, folds={item['num_folds']}"
        for item in summary_rows
    ]


def format_task_summary(summary: Dict[str, List[Dict[str, float]]]) -> List[str]:
    lines: List[str] = []
    for task, items in summary.items():
        formatted = " | ".join(
            [f"{item['model']}: f1={item['mean_f1']:.4f}, acc={item['mean_acc']:.4f}" for item in items]
        )
        lines.append(f"{task}: {formatted}")
    return lines


In [23]:
rows = build_result_rows(MODEL_RUNS, TASKS, NUM_FOLDS)
fold_ranking = build_fold_ranking(rows, NUM_FOLDS)
top3_lines = format_top_k(rows, top_k=3)
model_summary_lines = format_model_summary(build_model_summary(rows))
task_summary_lines = format_task_summary(task_level_summary(rows, TASKS))

print("=== Per-fold Ranking By Average Macro-F1 ===")
for line in format_fold_ranking(fold_ranking):
    print(line)

print("\n=== Global Top 3 ===")
for line in top3_lines:
    print(line)

print("\n=== Model Mean Over 10 Folds ===")
for line in model_summary_lines:
    print(line)

print("\n=== Task-level Mean Over 10 Folds ===")
for line in task_summary_lines:
    print(line)

analysis_report = {
    "rows": rows,
    "fold_ranking": fold_ranking,
    "top3": sorted(rows, key=lambda item: item["avg_f1"], reverse=True)[:3],
    "model_summary": build_model_summary(rows),
    "task_summary": task_level_summary(rows, TASKS),
}

analysis_report

=== Per-fold Ranking By Average Macro-F1 ===
fold_0: ssm: f1=0.3144, acc=0.7098 | st_gcn: f1=0.2129, acc=0.5180 | tcn: f1=0.1910, acc=0.4903
fold_1: tcn: f1=0.2284, acc=0.2986 | ssm: f1=0.1730, acc=0.3568 | st_gcn: f1=0.1481, acc=0.2473
fold_2: tcn: f1=0.2870, acc=0.5999 | ssm: f1=0.1464, acc=0.2856 | st_gcn: f1=0.1137, acc=0.2395
fold_3: tcn: f1=0.3065, acc=0.4425 | ssm: f1=0.2807, acc=0.4058 | st_gcn: f1=0.2351, acc=0.4606
fold_4: ssm: f1=0.3077, acc=0.5506 | tcn: f1=0.2485, acc=0.4042 | st_gcn: f1=0.2274, acc=0.5018
fold_5: ssm: f1=0.3670, acc=0.5634 | tcn: f1=0.3449, acc=0.5360 | st_gcn: f1=0.2414, acc=0.6004
fold_6: ssm: f1=0.3576, acc=0.6740 | tcn: f1=0.2424, acc=0.4876 | st_gcn: f1=0.1955, acc=0.4724
fold_7: ssm: f1=0.2441, acc=0.4326 | tcn: f1=0.1678, acc=0.3633 | st_gcn: f1=0.1445, acc=0.3018
fold_8: ssm: f1=0.2431, acc=0.4606 | st_gcn: f1=0.2021, acc=0.4447 | tcn: f1=0.2020, acc=0.4615
fold_9: tcn: f1=0.3057, acc=0.5780 | ssm: f1=0.2761, acc=0.5151 | st_gcn: f1=0.2240, acc=0.

{'rows': [{'model': 'ssm',
   'fold': 0,
   'avg_acc': 0.7097966745495796,
   'avg_f1': 0.31444496613267453,
   'tasks': {'twist': {'acc': 0.9852125644683838,
     'f1': 0.3308504034761018,
     'num_samples': 541},
    'posture': {'acc': 0.6894639730453491,
     'f1': 0.41817585301837273,
     'num_samples': 541},
    'relax': {'acc': 0.45471349358558655,
     'f1': 0.2279265719607156,
     'num_samples': 541},
    'total': {'acc': 0.709796667098999,
     'f1': 0.280827036075508,
     'num_samples': 541}}},
  {'model': 'ssm',
   'fold': 1,
   'avg_acc': 0.35681818798184395,
   'avg_f1': 0.17295575866634746,
   'tasks': {'twist': {'acc': 0.41090908646583557,
     'f1': 0.19415807560137457,
     'num_samples': 550},
    'posture': {'acc': 0.18727272748947144,
     'f1': 0.10515569167942829,
     'num_samples': 550},
    'relax': {'acc': 0.2309090942144394,
     'f1': 0.14298329089615322,
     'num_samples': 550},
    'total': {'acc': 0.5981818437576294,
     'f1': 0.2495259764884338,
  

In [24]:

# 分别显示每个fold的详细结果
def format_fold_detailed(fold_ranking: Dict[int, List[Dict[str, object]]], tasks: List[str]) -> None:
    """显示每个fold的详细结果"""
    for fold, items in fold_ranking.items():
        print(f"\n{'='*80}")
        print(f"fold_{fold} 详细结果：")
        print(f"{'='*80}")
        
        if not items:
            print(f"  无结果")
            continue
            
        for rank, item in enumerate(items, start=1):
            print(f"\n  第{rank}名: {item['model']}")
            print(f"    平均准确率 (avg_acc): {item['avg_acc']:.4f}")
            print(f"    平均F1分数 (avg_f1): {item['avg_f1']:.4f}")
            print(f"    各任务详情:")
            for task in tasks:
                if task in item['tasks']:
                    task_result = item['tasks'][task]
                    print(f"      {task:10s} - acc: {task_result['acc']:.4f}, f1: {task_result['f1']:.4f}, samples: {task_result['num_samples']}")

print("=" * 80)
print("每个Fold的详细结果")
print("=" * 80)
format_fold_detailed(fold_ranking, TASKS)


每个Fold的详细结果

fold_0 详细结果：

  第1名: ssm
    平均准确率 (avg_acc): 0.7098
    平均F1分数 (avg_f1): 0.3144
    各任务详情:
      twist      - acc: 0.9852, f1: 0.3309, samples: 541
      posture    - acc: 0.6895, f1: 0.4182, samples: 541
      relax      - acc: 0.4547, f1: 0.2279, samples: 541
      total      - acc: 0.7098, f1: 0.2808, samples: 541

  第2名: st_gcn
    平均准确率 (avg_acc): 0.5180
    平均F1分数 (avg_f1): 0.2129
    各任务详情:
      twist      - acc: 1.0000, f1: 0.3333, samples: 541
      posture    - acc: 0.4067, f1: 0.1927, samples: 541
      relax      - acc: 0.4455, f1: 0.2055, samples: 541
      total      - acc: 0.2200, f1: 0.1202, samples: 541

  第3名: tcn
    平均准确率 (avg_acc): 0.4903
    平均F1分数 (avg_f1): 0.1910
    各任务详情:
      twist      - acc: 1.0000, f1: 0.3333, samples: 541
      posture    - acc: 0.4067, f1: 0.1927, samples: 541
      relax      - acc: 0.5545, f1: 0.2378, samples: 541
      total      - acc: 0.0000, f1: 0.0000, samples: 541

fold_1 详细结果：

  第1名: tcn
    平均准确率 (avg_acc): 0.2

In [25]:

# 计算Top3的平均结果
def compute_top_k_average(rows: List[Dict[str, object]], tasks: List[str], top_k: int = 3) -> Dict[str, object]:
    """计算Top K结果的平均指标"""
    sorted_rows = sorted(rows, key=lambda item: item["avg_f1"], reverse=True)[:top_k]
    
    if not sorted_rows:
        return {}
    
    # 计算全局平均
    avg_acc = sum(item["avg_acc"] for item in sorted_rows) / len(sorted_rows)
    avg_f1 = sum(item["avg_f1"] for item in sorted_rows) / len(sorted_rows)
    
    # 计算各任务平均
    task_averages = {}
    for task in tasks:
        task_accs = []
        task_f1s = []
        for item in sorted_rows:
            if task in item["tasks"]:
                task_accs.append(item["tasks"][task]["acc"])
                task_f1s.append(item["tasks"][task]["f1"])
        if task_accs:
            task_averages[task] = {
                "avg_acc": sum(task_accs) / len(task_accs),
                "avg_f1": sum(task_f1s) / len(task_f1s),
            }
    
    return {
        "top_k": top_k,
        "count": len(sorted_rows),
        "avg_acc": avg_acc,
        "avg_f1": avg_f1,
        "task_averages": task_averages,
        "top_items": sorted_rows,
    }

# 输出Top3的平均结果
print("\n" + "=" * 80)
print("Top 3 的平均结果统计")
print("=" * 80)

top3_summary = compute_top_k_average(rows, TASKS, top_k=3)

print(f"\n总体平均指标 (Top 3平均):")
print(f"  平均准确率 (avg_acc): {top3_summary['avg_acc']:.4f}")
print(f"  平均F1分数 (avg_f1): {top3_summary['avg_f1']:.4f}")

print(f"\nTop 3 的组成 (按F1分数排序):")
for rank, item in enumerate(top3_summary['top_items'], start=1):
    print(f"  第{rank}名: {item['model']} fold_{item['fold']} | avg_f1: {item['avg_f1']:.4f}, avg_acc: {item['avg_acc']:.4f}")

print(f"\n各任务平均指标 (Top 3中):")
for task in TASKS:
    if task in top3_summary['task_averages']:
        metrics = top3_summary['task_averages'][task]
        print(f"  {task:10s}: acc={metrics['avg_acc']:.4f}, f1={metrics['avg_f1']:.4f}")



Top 3 的平均结果统计

总体平均指标 (Top 3平均):
  平均准确率 (avg_acc): 0.5911
  平均F1分数 (avg_f1): 0.3565

Top 3 的组成 (按F1分数排序):
  第1名: ssm fold_5 | avg_f1: 0.3670, avg_acc: 0.5634
  第2名: ssm fold_6 | avg_f1: 0.3576, avg_acc: 0.6740
  第3名: tcn fold_5 | avg_f1: 0.3449, avg_acc: 0.5360

各任务平均指标 (Top 3中):
  twist     : acc=0.8012, f1=0.4071
  posture   : acc=0.4960, f1=0.3208
  relax     : acc=0.4597, f1=0.3159
  total     : acc=0.6076, f1=0.3822


In [26]:
# 计算所有fold（全部模型-fold条目）的平均结果
def compute_all_folds_average(rows: List[Dict[str, object]], tasks: List[str], top_k: int = 3) -> Dict[str, object]:
    """计算所有fold的整体平均指标，并按backbone分组汇总与TopK统计"""
    if not rows:
        return {}

    overall_avg_acc = sum(item["avg_acc"] for item in rows) / len(rows)
    overall_avg_f1 = sum(item["avg_f1"] for item in rows) / len(rows)

    task_averages: Dict[str, Dict[str, float]] = {}
    for task in tasks:
        task_accs = [item["tasks"][task]["acc"] for item in rows if task in item["tasks"]]
        task_f1s = [item["tasks"][task]["f1"] for item in rows if task in item["tasks"]]
        if task_accs:
            task_averages[task] = {
                "avg_acc": float(sum(task_accs) / len(task_accs)),
                "avg_f1": float(sum(task_f1s) / len(task_f1s)),
            }

    fold_ids = sorted({int(item["fold"]) for item in rows})
    per_fold_average: Dict[int, Dict[str, float]] = {}
    for fold in fold_ids:
        fold_rows = [item for item in rows if int(item["fold"]) == fold]
        per_fold_average[fold] = {
            "avg_acc": float(sum(item["avg_acc"] for item in fold_rows) / len(fold_rows)),
            "avg_f1": float(sum(item["avg_f1"] for item in fold_rows) / len(fold_rows)),
        }

    # 按backbone（model字段）分组统计
    backbone_summary: Dict[str, Dict[str, object]] = {}
    backbone_topk_summary: Dict[str, Dict[str, object]] = {}
    backbone_names = sorted({str(item["model"]) for item in rows})
    for backbone in backbone_names:
        backbone_rows = [item for item in rows if str(item["model"]) == backbone]

        backbone_task_averages: Dict[str, Dict[str, float]] = {}
        for task in tasks:
            b_task_accs = [item["tasks"][task]["acc"] for item in backbone_rows if task in item["tasks"]]
            b_task_f1s = [item["tasks"][task]["f1"] for item in backbone_rows if task in item["tasks"]]
            if b_task_accs:
                backbone_task_averages[task] = {
                    "avg_acc": float(sum(b_task_accs) / len(b_task_accs)),
                    "avg_f1": float(sum(b_task_f1s) / len(b_task_f1s)),
                }

        backbone_summary[backbone] = {
            "count": len(backbone_rows),
            "avg_acc": float(sum(item["avg_acc"] for item in backbone_rows) / len(backbone_rows)),
            "avg_f1": float(sum(item["avg_f1"] for item in backbone_rows) / len(backbone_rows)),
            "task_averages": backbone_task_averages,
        }

        # 在每个backbone内部取Top K（按avg_f1）
        top_items = sorted(backbone_rows, key=lambda item: item["avg_f1"], reverse=True)[:top_k]
        if top_items:
            top_task_averages: Dict[str, Dict[str, float]] = {}
            for task in tasks:
                t_accs = [item["tasks"][task]["acc"] for item in top_items if task in item["tasks"]]
                t_f1s = [item["tasks"][task]["f1"] for item in top_items if task in item["tasks"]]
                if t_accs:
                    top_task_averages[task] = {
                        "avg_acc": float(sum(t_accs) / len(t_accs)),
                        "avg_f1": float(sum(t_f1s) / len(t_f1s)),
                    }

            backbone_topk_summary[backbone] = {
                "top_k": top_k,
                "count": len(top_items),
                "avg_acc": float(sum(item["avg_acc"] for item in top_items) / len(top_items)),
                "avg_f1": float(sum(item["avg_f1"] for item in top_items) / len(top_items)),
                "folds": [int(item["fold"]) for item in top_items],
                "top_items": [
                    {
                        "fold": int(item["fold"]),
                        "avg_acc": float(item["avg_acc"]),
                        "avg_f1": float(item["avg_f1"]),
                    }
                    for item in top_items
                ],
                "task_averages": top_task_averages,
            }

    return {
        "num_entries": len(rows),
        "num_folds": len(fold_ids),
        "overall_avg_acc": float(overall_avg_acc),
        "overall_avg_f1": float(overall_avg_f1),
        "task_averages": task_averages,
        "per_fold_average": per_fold_average,
        "backbone_summary": backbone_summary,
        "backbone_topk_summary": backbone_topk_summary,
    }


print("\n" + "=" * 80)
print("所有 Fold 的平均结果统计（按 Backbone 分组）")
print("=" * 80)

all_folds_summary = compute_all_folds_average(rows, TASKS, top_k=3)

if not all_folds_summary:
    print("没有可用结果")
else:
    print(f"\n样本条目数: {all_folds_summary['num_entries']} (模型-fold组合)")
    print(f"fold数量: {all_folds_summary['num_folds']}")
    print(f"\n总体平均准确率 (overall_avg_acc): {all_folds_summary['overall_avg_acc']:.4f}")
    print(f"总体平均F1分数 (overall_avg_f1): {all_folds_summary['overall_avg_f1']:.4f}")

    print("\n按 Backbone 的总体平均:")
    for backbone, metrics in all_folds_summary["backbone_summary"].items():
        print(
            f"  {backbone:10s}: avg_acc={metrics['avg_acc']:.4f}, "
            f"avg_f1={metrics['avg_f1']:.4f}, n={metrics['count']}"
        )

    print("\n按 Backbone 计算 Top 3（按各自backbone内的avg_f1排序）并取平均:")
    for backbone, metrics in all_folds_summary["backbone_topk_summary"].items():
        fold_text = ", ".join([f"fold_{f}" for f in metrics["folds"]])
        print(
            f"  {backbone:10s}: top3_avg_acc={metrics['avg_acc']:.4f}, "
            f"top3_avg_f1={metrics['avg_f1']:.4f}, folds=[{fold_text}]"
        )
        for item in metrics["top_items"]:
            print(
                f"    - fold_{item['fold']}: avg_f1={item['avg_f1']:.4f}, "
                f"avg_acc={item['avg_acc']:.4f}"
            )

    print("\n各fold平均（先对同一fold的模型结果求均值）:")
    for fold, metrics in all_folds_summary["per_fold_average"].items():
        print(f"  fold_{fold}: avg_acc={metrics['avg_acc']:.4f}, avg_f1={metrics['avg_f1']:.4f}")

    # 分别写入“总体结果”和“Top3平均结果”
    import json

    output_dir = Path("/workspace/code/project/analysis/output")
    output_dir.mkdir(parents=True, exist_ok=True)

    overall_result_to_save = {
        "num_entries": all_folds_summary["num_entries"],
        "num_folds": all_folds_summary["num_folds"],
        "overall_avg_acc": all_folds_summary["overall_avg_acc"],
        "overall_avg_f1": all_folds_summary["overall_avg_f1"],
        "task_averages": all_folds_summary["task_averages"],
        "per_fold_average": all_folds_summary["per_fold_average"],
        "backbone_summary": all_folds_summary["backbone_summary"],
    }

    top3_result_to_save = all_folds_summary["backbone_topk_summary"]

    overall_path = output_dir / "overall_result.json"
    top3_path = output_dir / "backbone_top3_result.json"

    overall_path.write_text(
        json.dumps(overall_result_to_save, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    top3_path.write_text(
        json.dumps(top3_result_to_save, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    print("\n结果文件已保存:")
    print(f"  - 总体结果: {overall_path}")
    print(f"  - Top3平均结果: {top3_path}")

all_folds_summary


所有 Fold 的平均结果统计（按 Backbone 分组）

样本条目数: 30 (模型-fold组合)
fold数量: 10

总体平均准确率 (overall_avg_acc): 0.4639
总体平均F1分数 (overall_avg_f1): 0.2393

按 Backbone 的总体平均:
  ssm       : avg_acc=0.4954, avg_f1=0.2710, n=10
  st_gcn    : avg_acc=0.4301, avg_f1=0.1945, n=10
  tcn       : avg_acc=0.4662, avg_f1=0.2524, n=10

按 Backbone 计算 Top 3（按各自backbone内的avg_f1排序）并取平均:
  ssm       : top3_avg_acc=0.6491, top3_avg_f1=0.3463, folds=[fold_5, fold_6, fold_0]
    - fold_5: avg_f1=0.3670, avg_acc=0.5634
    - fold_6: avg_f1=0.3576, avg_acc=0.6740
    - fold_0: avg_f1=0.3144, avg_acc=0.7098
  st_gcn    : top3_avg_acc=0.5209, top3_avg_f1=0.2346, folds=[fold_5, fold_3, fold_4]
    - fold_5: avg_f1=0.2414, avg_acc=0.6004
    - fold_3: avg_f1=0.2351, avg_acc=0.4606
    - fold_4: avg_f1=0.2274, avg_acc=0.5018
  tcn       : top3_avg_acc=0.5188, top3_avg_f1=0.3190, folds=[fold_5, fold_3, fold_9]
    - fold_5: avg_f1=0.3449, avg_acc=0.5360
    - fold_3: avg_f1=0.3065, avg_acc=0.4425
    - fold_9: avg_f1=0.3057, avg_acc=

{'num_entries': 30,
 'num_folds': 10,
 'overall_avg_acc': 0.46390060503035785,
 'overall_avg_f1': 0.23929859517799434,
 'task_averages': {'twist': {'avg_acc': 0.6493272741635641,
   'avg_f1': 0.27655897502919324},
  'posture': {'avg_acc': 0.44571583718061447, 'avg_f1': 0.2458180692671216},
  'relax': {'avg_acc': 0.38591712738076844, 'avg_f1': 0.21786991794184604},
  'total': {'avg_acc': 0.37464218139648436, 'avg_f1': 0.2169474184738163}},
 'per_fold_average': {0: {'avg_acc': 0.5727048702538013,
   'avg_f1': 0.2394478755296028},
  1: {'avg_acc': 0.3009090907871723, 'avg_f1': 0.18314587848102928},
  2: {'avg_acc': 0.37500000186264515, 'avg_f1': 0.1823634383640854},
  3: {'avg_acc': 0.4363017976284027, 'avg_f1': 0.27406407986334097},
  4: {'avg_acc': 0.4855535353223483, 'avg_f1': 0.26119797479352425},
  5: {'avg_acc': 0.566605843603611, 'avg_f1': 0.31779013531549577},
  6: {'avg_acc': 0.5446768129865328, 'avg_f1': 0.265162862693975},
  7: {'avg_acc': 0.3659021419783433, 'avg_f1': 0.185502

In [27]:
# 增强导出：总计结果中写入所有精度明细（每个模型/每个fold/每个任务）
import json

output_dir = Path("/workspace/code/project/analysis/output")
output_dir.mkdir(parents=True, exist_ok=True)

if not rows:
    print("没有可导出的 rows 数据，请先运行第2个和第5个单元")
else:
    # 每条记录的完整精度明细
    all_entry_details = []
    for item in rows:
        task_acc = {task: float(item["tasks"][task]["acc"]) for task in TASKS if task in item["tasks"]}
        task_f1 = {task: float(item["tasks"][task]["f1"]) for task in TASKS if task in item["tasks"]}
        all_entry_details.append(
            {
                "model": str(item["model"]),
                "fold": int(item["fold"]),
                "avg_acc": float(item["avg_acc"]),
                "avg_f1": float(item["avg_f1"]),
                "task_acc": task_acc,
                "task_f1": task_f1,
            }
        )

    # 聚合所有精度值，便于后处理（作图/统计）
    all_accuracy_values = {
        "avg_acc": [float(item["avg_acc"]) for item in rows],
        "by_task_acc": {
            task: [float(item["tasks"][task]["acc"]) for item in rows if task in item["tasks"]]
            for task in TASKS
        },
    }

    enhanced_overall_result = {
        "summary": {
            "num_entries": int(all_folds_summary.get("num_entries", len(rows))) if isinstance(all_folds_summary, dict) else len(rows),
            "num_folds": int(all_folds_summary.get("num_folds", len(set(int(item["fold"]) for item in rows)))) if isinstance(all_folds_summary, dict) else len(set(int(item["fold"]) for item in rows)),
            "overall_avg_acc": float(all_folds_summary.get("overall_avg_acc", 0.0)) if isinstance(all_folds_summary, dict) else 0.0,
            "overall_avg_f1": float(all_folds_summary.get("overall_avg_f1", 0.0)) if isinstance(all_folds_summary, dict) else 0.0,
        },
        "all_entry_details": all_entry_details,
        "all_accuracy_values": all_accuracy_values,
        "task_averages": all_folds_summary.get("task_averages", {}) if isinstance(all_folds_summary, dict) else {},
        "per_fold_average": all_folds_summary.get("per_fold_average", {}) if isinstance(all_folds_summary, dict) else {},
        "backbone_summary": all_folds_summary.get("backbone_summary", {}) if isinstance(all_folds_summary, dict) else {},
        "backbone_topk_summary": all_folds_summary.get("backbone_topk_summary", {}) if isinstance(all_folds_summary, dict) else {},
    }

    enhanced_overall_path = output_dir / "overall_result_with_all_accuracy.json"
    enhanced_overall_path.write_text(
        json.dumps(enhanced_overall_result, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    print("增强总计结果已保存:")
    print(f"  - {enhanced_overall_path}")

增强总计结果已保存:
  - /workspace/code/project/analysis/output/overall_result_with_all_accuracy.json


In [28]:
# 增强导出：Top K 文件写入完整结果（含每个入选条目的全部任务指标与全量排序）
import json

output_dir = Path("/workspace/code/project/analysis/output")
output_dir.mkdir(parents=True, exist_ok=True)

if not rows:
    print("没有可导出的 rows 数据，请先运行第2个和第5个单元")
else:
    top_k = 3
    enriched_topk_by_backbone = {}

    backbone_names = sorted({str(item["model"]) for item in rows})
    for backbone in backbone_names:
        backbone_rows = [item for item in rows if str(item["model"]) == backbone]
        sorted_rows = sorted(backbone_rows, key=lambda x: x["avg_f1"], reverse=True)
        top_items = sorted_rows[:top_k]

        # Top K 入选条目的完整信息（含各任务acc/f1/samples）
        top_items_full = []
        for item in top_items:
            top_items_full.append(
                {
                    "model": str(item["model"]),
                    "fold": int(item["fold"]),
                    "avg_acc": float(item["avg_acc"]),
                    "avg_f1": float(item["avg_f1"]),
                    "tasks": {
                        task: {
                            "acc": float(item["tasks"][task]["acc"]),
                            "f1": float(item["tasks"][task]["f1"]),
                            "num_samples": int(item["tasks"][task]["num_samples"]),
                        }
                        for task in TASKS
                        if task in item["tasks"]
                    },
                }
            )

        # 该 backbone 的全量排序结果（用于核对 Top K 来源）
        all_ranked_items = [
            {
                "model": str(item["model"]),
                "fold": int(item["fold"]),
                "avg_acc": float(item["avg_acc"]),
                "avg_f1": float(item["avg_f1"]),
            }
            for item in sorted_rows
        ]

        if top_items:
            # 兼容旧口径：先对每个条目的avg_acc/avg_f1求平均
            topk_avg_from_entry = {
                "avg_acc": float(sum(item["avg_acc"] for item in top_items) / len(top_items)),
                "avg_f1": float(sum(item["avg_f1"] for item in top_items) / len(top_items)),
            }

            # 新口径：对所有metrics（各任务acc/f1/num_samples）做平均
            task_metric_average = {}
            for task in TASKS:
                task_rows = [item for item in top_items if task in item["tasks"]]
                if not task_rows:
                    continue
                task_metric_average[task] = {
                    "acc": float(sum(item["tasks"][task]["acc"] for item in task_rows) / len(task_rows)),
                    "f1": float(sum(item["tasks"][task]["f1"] for item in task_rows) / len(task_rows)),
                    "num_samples": float(sum(item["tasks"][task]["num_samples"] for item in task_rows) / len(task_rows)),
                }

            flat_acc_values = [
                float(item["tasks"][task]["acc"])
                for item in top_items
                for task in TASKS
                if task in item["tasks"]
            ]
            flat_f1_values = [
                float(item["tasks"][task]["f1"])
                for item in top_items
                for task in TASKS
                if task in item["tasks"]
            ]
            flat_num_samples = [
                float(item["tasks"][task]["num_samples"])
                for item in top_items
                for task in TASKS
                if task in item["tasks"]
            ]

            topk_avg_all_metrics = {
                "acc": float(sum(flat_acc_values) / len(flat_acc_values)) if flat_acc_values else 0.0,
                "f1": float(sum(flat_f1_values) / len(flat_f1_values)) if flat_f1_values else 0.0,
                "num_samples": float(sum(flat_num_samples) / len(flat_num_samples)) if flat_num_samples else 0.0,
            }
        else:
            topk_avg_from_entry = {"avg_acc": 0.0, "avg_f1": 0.0}
            task_metric_average = {}
            topk_avg_all_metrics = {"acc": 0.0, "f1": 0.0, "num_samples": 0.0}

        enriched_topk_by_backbone[backbone] = {
            "top_k": top_k,
            "count": len(top_items),
            # 新默认口径：Top K 内所有metrics平均
            "topk_avg_acc": topk_avg_all_metrics["acc"],
            "topk_avg_f1": topk_avg_all_metrics["f1"],
            "topk_avg_num_samples": topk_avg_all_metrics["num_samples"],
            # 旧口径保留，便于对照
            "topk_avg_from_entry": topk_avg_from_entry,
            # 任务级平均（Top K）
            "topk_task_metric_average": task_metric_average,
            "top_folds": [int(item["fold"]) for item in top_items],
            "top_items_full": top_items_full,
            "all_ranked_items": all_ranked_items,
        }

    topk_export = {
        "description": "Top K per backbone with full task metrics, full ranking list, and all-metrics averages",
        "backbone_topk_full": enriched_topk_by_backbone,
    }

    # 覆盖原 Top K 文件，使其包含完整结果
    topk_full_path = output_dir / "backbone_top3_result.json"
    topk_full_path.write_text(
        json.dumps(topk_export, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    print("Top K 完整结果已保存:")
    print(f"  - {topk_full_path}")

Top K 完整结果已保存:
  - /workspace/code/project/analysis/output/backbone_top3_result.json
